# Data poisoning & backdoors

Poisoning changes training data so the learned model behaves badly later, often only when a trigger appears.

Gap note: the lesson body is thin, so the notebook anchors to the plan's stable toy numbers and implements a compact backdoor simulation with trigger-conditional evaluation. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 19
rng = np.random.default_rng(SEED)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def make_group(X, y):
    scores = X[:, 0]
    if len(np.unique(scores)) < 3:
        scores = X.sum(axis=1)
    cutoff = np.median(scores)
    group = (scores > cutoff).astype(int)
    if len(np.unique(group)) < 2:
        group = (np.arange(len(y)) % 2).astype(int)
    return group


def safe_split(X, y, group=None, test_size=0.4):
    stratify = y
    if min(np.bincount(y.astype(int))) < 2:
        stratify = None
    pieces = train_test_split(
        X,
        y,
        np.arange(len(y)),
        test_size=test_size,
        random_state=0,
        stratify=stratify,
    )
    x_train, x_test, y_train, y_test, idx_train, idx_test = pieces
    if group is None:
        return x_train, x_test, y_train, y_test, idx_train, idx_test, None, None
    return x_train, x_test, y_train, y_test, idx_train, idx_test, group[idx_train], group[idx_test]


def fit_scaled_logreg(X, y, C=1.0):
    x_train, x_test, y_train, y_test, idx_train, idx_test, _, _ = safe_split(X, y)
    scaler = StandardScaler()
    x_train_s = scaler.fit_transform(x_train)
    x_test_s = scaler.transform(x_test)
    model = LogisticRegression(max_iter=2000, C=C, multi_class="auto")
    model.fit(x_train_s, y_train)
    return model, scaler, x_train_s, x_test_s, y_train, y_test, idx_train, idx_test


def probability_for_label(model, X, y):
    probs = model.predict_proba(X)
    positions = np.array([np.where(model.classes_ == label)[0][0] for label in y])
    return probs[np.arange(len(y)), positions]


def fgsm_attack(model, X, y, eps):
    probs = model.predict_proba(X)
    class_positions = np.array([np.where(model.classes_ == label)[0][0] for label in y])
    one_hot = np.zeros_like(probs)
    one_hot[np.arange(len(y)), class_positions] = 1.0
    grad = (probs - one_hot) @ model.coef_
    direction = np.sign(grad)
    return X + eps * direction


def robust_accuracy_for_eps(model, X, y, eps):
    attacked = fgsm_attack(model, X, y, eps)
    preds = model.predict(attacked)
    return accuracy_score(y, preds)


def fairness_report(y, yhat, group):
    rows = {}
    positive = int(np.max(y))
    for g in [0, 1]:
        mask = group == g
        truth_pos = mask & (y == positive)
        truth_neg = mask & (y != positive)
        pred_pos = mask & (yhat == positive)
        tp = int(np.sum(pred_pos & truth_pos))
        fp = int(np.sum(pred_pos & truth_neg))
        fn = int(np.sum((~pred_pos) & truth_pos))
        tn = int(np.sum((~pred_pos) & truth_neg))
        rate = float(np.mean(yhat[mask] == positive)) if np.any(mask) else np.nan
        tpr = tp / max(tp + fn, 1)
        fpr = fp / max(fp + tn, 1)
        rows[g] = {"n": int(mask.sum()), "pos_rate": rate, "tpr": tpr, "fpr": fpr, "tp": tp, "fp": fp, "fn": fn, "tn": tn}
    dp_gap = abs(rows[0]["pos_rate"] - rows[1]["pos_rate"])
    eo_gap = max(abs(rows[0]["tpr"] - rows[1]["tpr"]), abs(rows[0]["fpr"] - rows[1]["fpr"]))
    return {"group0": rows[0], "group1": rows[1], "dp_gap": dp_gap, "eo_gap": eo_gap}


def plot_2d_projection(ax, X, color, title):
    if X.shape[1] >= 2:
        shown = X[:, :2]
    else:
        shown = np.column_stack([X[:, 0], np.zeros(len(X))])
    ax.scatter(shown[:, 0], shown[:, 1], c=color, s=18, cmap="viridis", alpha=0.75)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once: poisoned risk mixture

The lesson formula is $$R_{poison}=(1-\alpha)R_{clean}+\alpha R_{bad}.$$ Gap note: the lesson body is thin, so the plan's stable toy numbers anchor the assertions.

In [ ]:

components = np.array([0.05, 0.4, 0.12], dtype=float)
knob = 0.08
total = float(np.sum(components))
absolute_total = float(np.sum(np.abs(components)))
leading_share = float(abs(components[0]) / absolute_total)
guarded = float(total + knob * absolute_total)
contrast = float(total - components[2])
change = float(contrast - total)

assert np.isclose(total, 0.57)
assert np.isclose(absolute_total, 0.57)
assert np.isclose(round(guarded, 3), 0.616)
assert np.isclose(round(leading_share, 3), 0.088)
assert np.isclose(knob, 0.080)

print("total", round(total, 3))
print("absolute_total", round(absolute_total, 3))
print("leading_share", round(leading_share, 3))
print("guarded", round(guarded, 3))
print("change_without_component_3", round(change, 3))


The concrete backdoor adds a trigger offset to selected training rows and relabels them to a target class. Evaluation must include both clean and triggered slices.

In [ ]:

def add_trigger(X, strength=4.0):
    triggered = X.copy()
    triggered[:, -1] = triggered[:, -1] + strength
    return triggered


def poison_train_eval(X, y, alpha=0.05, trigger_fn=add_trigger):
    x_train, x_test, y_train, y_test, idx_train, idx_test, _, _ = safe_split(X, y)
    target = int(np.max(y))
    poison_count = max(1, int(alpha * len(x_train))) if alpha > 0 else 0
    poisoned_x = x_train.copy()
    poisoned_y = y_train.copy()
    if poison_count > 0:
        chosen = np.arange(poison_count)
        poisoned_x[chosen] = trigger_fn(poisoned_x[chosen])
        poisoned_y[chosen] = target
    scaler = StandardScaler()
    poisoned_x_s = scaler.fit_transform(poisoned_x)
    x_test_s = scaler.transform(x_test)
    triggered_test_s = scaler.transform(trigger_fn(x_test))
    model = LogisticRegression(max_iter=2000, multi_class="auto")
    model.fit(poisoned_x_s, poisoned_y)
    clean_acc = accuracy_score(y_test, model.predict(x_test_s))
    trigger_target_rate = float(np.mean(model.predict(triggered_test_s) == target))
    robust_metric = clean_acc - trigger_target_rate
    return {"model": model, "clean_acc": clean_acc, "trigger_target_rate": trigger_target_rate, "robust_metric": robust_metric, "poison_count": poison_count}

X, y = clf_ladder()[1][1:]
result = poison_train_eval(X, y, alpha=0.08)
print("poison_count", result["poison_count"])
print("clean_acc", round(result["clean_acc"], 3))
print("trigger_target_rate", round(result["trigger_target_rate"], 3))


## The dataset ladder

The same classifier family is tested on D1-D5: a hand toy, synthetic blobs, noisy moons, real Wine data, and real Breast Cancer data.

In [ ]:

rungs = clf_ladder()
for name, X, y in rungs:
    classes, counts = np.unique(y, return_counts=True)
    print(name)
    print("  shape", X.shape)
    print("  classes", dict(zip(classes.tolist(), counts.tolist())))
    print("  sample", np.round(X[:3, :min(4, X.shape[1])], 3))


## Run the same poisoning evaluation across D1-D5

The metric is clean accuracy minus triggered target rate. Larger is safer because clean performance remains useful while the trigger does not control predictions.

In [ ]:

poison_alpha = 0.05
poison_results = []
for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
    result = poison_train_eval(X, y, alpha=poison_alpha)
    poison_results.append({"rung": rung, "name": name, "clean_acc": result["clean_acc"], "trigger_target_rate": result["trigger_target_rate"], "robust_metric": result["robust_metric"]})
print("rung | clean_acc | trigger_target_rate | robust_metric")
for row in poison_results:
    print(row["rung"], round(row["clean_acc"], 3), round(row["trigger_target_rate"], 3), round(row["robust_metric"], 3))


## Results visualization

Left: clean and trigger rates per rung. Right: triggered failure versus poison rate.

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, row in zip(axes, poison_results):
    ax.bar(["clean acc", "trigger target"], [row["clean_acc"], row["trigger_target_rate"]], color=["tab:blue", "tab:red"])
    ax.set_ylim(0, 1.05)
    ax.set_title(f"D{row['rung']}")
plt.tight_layout()
plt.show()

alphas = [0.0, 0.02, 0.05, 0.10]
fig, ax = plt.subplots(figsize=(7, 4))
for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
    rates = [poison_train_eval(X, y, alpha=a)["trigger_target_rate"] for a in alphas]
    ax.plot(alphas, rates, marker="o", label=f"D{rung}")
ax.set_xlabel("poison rate alpha")
ax.set_ylabel("triggered target rate")
ax.legend()
plt.show()


## Pitfall on D5: checking only clean accuracy

The wrong report misses the conditional trigger. The fix reports clean, triggered, and simple influence-style review of the largest training losses.

In [ ]:

name, X, y = clf_ladder()[-1]
clean_only = poison_train_eval(X, y, alpha=0.05)
print("wrong_clean_accuracy_only", round(clean_only["clean_acc"], 3))
print("fixed_trigger_target_rate", round(clean_only["trigger_target_rate"], 3))
model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
train_confidence = probability_for_label(model, x_train, y_train)
loss_proxy = -np.log(np.clip(train_confidence, 1e-9, 1.0))
review = np.argsort(loss_proxy)[-5:]
print("influence_style_high_loss_indices", review.tolist())
print("high_loss_values", np.round(loss_proxy[review], 3).tolist())


## Evaluate it + Practice

- Metric: clean accuracy minus triggered target rate.
- No-skill baseline: clean accuracy alone.
- Cheap sanity check: alpha zero should reduce trigger control.
- Ablation: remove the trigger feature and check whether attack success falls.
- Failure signals: aggregate validation is high while a trigger slice is near target-class certainty.

Practice prompts:
1. Change one stress knob and predict the direction of the metric before running.


2. Change the trigger column and compare D4 versus D5 sensitivity.

3. Plot high-loss training examples before and after poisoning.